# 01. סביבת עבודה, התקנות ושחזוריות

**????.** ???? ???? ??? ? ?'??? ????? ? ????? ?????

**מטרה.** מוודאת שהריפוזיטורי, הנתונים והתלויות זמינים לפני כל ניתוח.

**מקום בתהליך.** מחברת 1 מתוך 9 בסדרת המחקר. יש להריץ את המחברות לפי הסדר המופיע ב-`notebooks/README.md`; עם זאת, קוד האתחול מאתר את שורש הפרויקט גם בהרצה עצמאית.

**קלטים.** תיקיית הפרויקט, requirements.txt וקובצי data/.

**פלטים.** גרסאות חבילות, מלאי נתונים ובדיקות קדם להרצה.

**קשר לשאלות המחקר.** כל שאלות המחקר: זהו בסיס השחזוריות המשותף ל-RQ1-RQ4.

**עקרונות שחזוריות.** `random_state` קבוע היכן שנדרש, כל התוצאות נשמרות תחת `outputs/`, מאפיינים מבוססי תווית נבנים מתוויות train בלבד, וכל קוד המקור הרלוונטי מוצג בתאי קוד עם נתיב ושורות מקור.

## אתחול בטוח ובלתי תלוי בתיקיית ההרצה

התא הבא מאתר את שורש הריפוזיטורי דרך `requirements.txt`, עובר אליו, מוסיף את `src` לנתיב הייבוא ומגדיר תצוגה עקבית. שינוי `__name__` מונע מבלוקי `if __name__ == "__main__"` שהועתקו כלשונם לפעול לפני תא ההרצה המתועד שלהם. הפלט הצפוי הוא הנתיב המוחלט של הפרויקט.

In [1]:
from pathlib import Path
import os
import sys
import warnings

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("לא נמצא שורש הפרויקט המכיל requirements.txt ו-data/")

ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "research_notebook_01"
print(f"Project root: {ROOT}")

Project root: C:\Users\Jimi\Desktop\github-social-network


**פלט ומשמעות.** הופעת הנתיב `github-social-network` מאשרת שכל הנתיבים היחסיים בהמשך יופנו לפרויקט הנכון. אם התא נכשל, אין להמשיך: הכשל מצביע על הרצה מחוץ לעץ הפרויקט או על עותק חסר.

## בדיקת תלויות והתקנה מותנית

התא בודק את החבילות המופיעות ב-`requirements.txt`. רק אם חסרה תלות הוא מפעיל `pip install -r requirements.txt`, ולכן הוא מתאים גם לסביבה חדשה וגם להרצה חוזרת. הפלט המצופה הוא `All required imports are available` או פירוט ההתקנה שבוצעה.

In [2]:
import importlib.util
import subprocess

required_imports = {
    "pandas": "pandas", "numpy": "numpy", "networkx": "networkx",
    "matplotlib": "matplotlib", "sklearn": "scikit-learn",
    "seaborn": "seaborn", "community": "python-louvain",
    "scipy": "scipy", "gensim": "gensim", "powerlaw": "powerlaw",
    "nbformat": "nbformat", "nbconvert": "nbconvert", "docx": "python-docx",
    "pypdf": "pypdf", "bidi": "python-bidi", "reportlab": "reportlab",
}
missing = [package for module, package in required_imports.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("All required imports are available")

All required imports are available


**פלט ומשמעות.** הודעת הזמינות מאשרת שאין צורך בתיקון ידני. התקנה מותנית שומרת על עצמאות המחברות בלי להתקין מחדש בכל ריצה.

## גרסאות, קובצי נתונים וחתימות

התא מתעד גרסאות חבילות, גודל קבצים ו-SHA-256. החתימות מאפשרות לזהות שינוי בנתונים שעלול להסביר תוצאה שונה. צפויות שלוש רשומות נתוני MUSAE ורשומת גרסה לכל חבילה מרכזית.

In [3]:
import hashlib
import importlib.metadata as metadata
import pandas as pd

packages = ["pandas", "numpy", "networkx", "matplotlib", "scikit-learn", "seaborn", "python-louvain", "gensim", "powerlaw"]
versions = pd.DataFrame({"package": packages, "version": [metadata.version(p) for p in packages]})
display(versions)

data_rows = []
for path in sorted((ROOT / "data").glob("musae_git_*")):
    data_rows.append({
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
data_inventory = pd.DataFrame(data_rows)
display(data_inventory)

,package,version
0,pandas,2.2.3
1,numpy,1.26.4
2,networkx,3.6.1
3,matplotlib,3.10.0
4,scikit-learn,1.6.0
5,seaborn,0.13.2
6,python-louvain,0.16
7,gensim,4.4.0
8,powerlaw,2.0.0


,file,bytes,sha256
0,musae_git_edges.csv,3595143,d0d6a4f16e6ce841ddcdd0ff3aac49a9d4da157ba0dea8...
1,musae_git_features.json,4380176,47761f93f839d84b213e9f2fa43c056761734b295d6b2e...
2,musae_git_target.csv,714229,99ab263ad81c342fea4cd141c79f3094e35a670af458f3...


**פלט ומשמעות.** טבלאות הגרסאות והחתימות הן provenance של ההרצה. צד שלישי יכול להשוות אותן לפני פירוש פערים מספריים, במיוחד ב-Word2Vec ובאלגוריתמים סטוכסטיים.

## סיכום המחברת

**מה בוצע.** אותר שורש הפרויקט, נבדקו/הותקנו תלויות ותועדו גרסאות וחתימות נתונים.

**ממצאים מרכזיים.** סביבת ההרצה שלמה ושלושת קובצי MUSAE נגישים ישירות, בהתאם לדרישת ה-PDF.

**חיבור למחברת הבאה.** מחברת 02 טוענת ומאמתת את הנתונים ובונה את אובייקט הגרף.